## 1. Import libraries

In [17]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


In [18]:
Path("student_folder")

WindowsPath('student_folder')

## 2. Configure input and output paths

Change `INPUT_DIR` if your extracted dataset is somewhere else. For example, on Windows you can use a raw path such as `r"C:\Users\YourName\Downloads\Student folder"`.


In [27]:
INPUT_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")

assert INPUT_DIR.exists(), f"Input folder not found: {INPUT_DIR.resolve()}"
assert (INPUT_DIR / "train").exists(), "Expected a train/ folder inside INPUT_DIR"
assert (INPUT_DIR / "student_test").exists(), "Expected a student_test/ folder inside INPUT_DIR"

print("Input :", INPUT_DIR.resolve())
print("Output:", OUTPUT_DIR.resolve())


Input : C:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\raw
Output: C:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed


In [28]:
def read_table(path, columns, dtypes):
    """Read a whitespace-separated, headerless PathSnap file with explicit types."""
    return pd.read_csv(
        path,
        sep=r"\s+",          # One backslash: split on any whitespace
        engine="python",
        header=None,
        names=columns,
        dtype=dtypes
    )


In [29]:
def process_trip(partition, trip_dir, output_dir):
    trip_id = trip_dir.name
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load raw files
    track = read_table(
        trip_dir / f"{trip_id}.track",
        ["longitude", "latitude", "timestamp_s"],
        {"longitude": "float64", "latitude": "float64", "timestamp_s": "float64"}
    )
    nodes = read_table(
        trip_dir / f"{trip_id}.nodes",
        ["longitude", "latitude"],
        {"longitude": "float64", "latitude": "float64"}
    )
    arcs = read_table(
        trip_dir / f"{trip_id}.arcs",
        ["from_node_id", "to_node_id"],
        {"from_node_id": "int64", "to_node_id": "int64"}
    )
    route = read_table(
        trip_dir / f"{trip_id}.route",
        ["arc_id"],
        {"arc_id": "int64"}
    )

    original_gps_rows = len(track)

    # ---- GPS cleaning ----
    track = track.replace([np.inf, -np.inf], np.nan)
    track = track.dropna(
        subset=["longitude", "latitude", "timestamp_s"]
    )

    track = track[
        track["longitude"].between(-180, 180)
        & track["latitude"].between(-90, 90)
    ].copy()

    # Keep original observation order; do not silently reorder GPS points.
    timestamps = track["timestamp_s"].to_numpy(dtype=float)
    dt = np.diff(timestamps)

    # ---- Road-network reference checks ----
    # Keep every node and arc row in its original order.
    bad_arc_nodes = (
        ~arcs["from_node_id"].between(0, len(nodes) - 1)
        | ~arcs["to_node_id"].between(0, len(nodes) - 1)
    )

    # ---- Route reference checks ----
    bad_route_arcs = ~route["arc_id"].between(0, len(arcs) - 1)

    # ---- Export CSV files ----
    track.to_csv(output_dir / "track.csv", index_label="observation_id")
    nodes.to_csv(output_dir / "nodes.csv", index_label="node_id")
    arcs.to_csv(output_dir / "arcs.csv", index_label="arc_id")
    route.to_csv(output_dir / "route.csv", index_label="route_step")

    # ---- Per-trip quality report ----
    return {
        "partition": partition,
        "trip_id": trip_id,
        "gps_observations": len(track),
        "gps_rows_removed": original_gps_rows - len(track),
        "road_nodes": len(nodes),
        "road_arcs": len(arcs),
        "reference_route_steps": len(route),
        "non_increasing_timestamp_steps": int((dt <= 0).sum()),
        "non_1s_intervals": int((np.abs(dt - 1.0) > 1e-9).sum()),
        "max_gap_s": float(np.max(dt)) if len(dt) else np.nan,
        "invalid_arc_node_references": int(bad_arc_nodes.sum()),
        "invalid_route_arc_references": int(bad_route_arcs.sum()),
        "longitude_min": float(track["longitude"].min()) if len(track) else np.nan,
        "longitude_max": float(track["longitude"].max()) if len(track) else np.nan,
        "latitude_min": float(track["latitude"].min()) if len(track) else np.nan,
        "latitude_max": float(track["latitude"].max()) if len(track) else np.nan,
    }


In [30]:
report_rows = []

for partition in ["train", "student_test"]:
    partition_dir = INPUT_DIR / partition

    if not partition_dir.exists():
        raise FileNotFoundError(f"Missing partition: {partition_dir}")

    trip_dirs = sorted(path for path in partition_dir.iterdir() if path.is_dir())

    print(f"Processing {partition}: {len(trip_dirs)} trips")

    for trip_dir in trip_dirs:
        trip_report = process_trip(
            partition=partition,
            trip_dir=trip_dir,
            output_dir=OUTPUT_DIR / partition / trip_dir.name
        )
        report_rows.append(trip_report)

inventory = pd.DataFrame(report_rows)

print(f"Finished processing {len(inventory)} trips.")
display(inventory.head())


Processing train: 80 trips
Processing student_test: 10 trips
Finished processing 90 trips.


,partition,trip_id,gps_observations,gps_rows_removed,road_nodes,road_arcs,reference_route_steps,non_increasing_timestamp_steps,non_1s_intervals,max_gap_s,invalid_arc_node_references,invalid_route_arc_references,longitude_min,longitude_max,latitude_min,latitude_max
0,train,00000001,1315,0,88809,165790,525,0,25,16.0,0,0,8.570524,8.647860,49.844053,50.211882
1,train,00000002,7243,0,7936,16139,262,0,184,160.0,0,0,10.999961,11.050902,47.658344,47.695459
2,train,00000003,821,0,9138,18028,79,0,3,2.0,0,0,37.154202,37.201595,55.977135,56.007585
3,train,00000004,1064,0,7350,15499,320,0,491,51.0,0,0,12.342280,12.459615,52.590308,52.718960
4,train,00000005,1045,0,898,1579,105,0,0,1.0,0,0,35.485577,35.544315,49.357581,49.613085


In [31]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

inventory_path = OUTPUT_DIR / "dataset_inventory.csv"
inventory.to_csv(inventory_path, index=False)

print("Saved:", inventory_path.resolve())


Saved: C:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\dataset_inventory.csv


In [32]:
partition_summary = inventory.groupby("partition").agg(
    trips=("trip_id", "count"),
    gps_observations=("gps_observations", "sum"),
    median_gps_per_trip=("gps_observations", "median"),
    road_nodes=("road_nodes", "sum"),
    road_arcs=("road_arcs", "sum"),
    reference_route_steps=("reference_route_steps", "sum"),
    gps_rows_removed=("gps_rows_removed", "sum"),
    non_1s_intervals=("non_1s_intervals", "sum"),
    max_gap_s=("max_gap_s", "max"),
)

summary_path = OUTPUT_DIR / "partition_summary.csv"
partition_summary.to_csv(summary_path)

display(partition_summary)
print("Saved:", summary_path.resolve())


,trips,gps_observations,median_gps_per_trip,road_nodes,road_arcs,reference_route_steps,gps_rows_removed,non_1s_intervals,max_gap_s
partition,,,,,,,,,
student_test,10,19160,1420.0,90689,185041,3477,0,264,210.0
train,80,208417,1529.0,1933485,3876904,31336,0,7284,153064.0


Saved: C:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\partition_summary.csv


In [33]:
quality_columns = [
    "partition",
    "trip_id",
    "gps_rows_removed",
    "non_increasing_timestamp_steps",
    "non_1s_intervals",
    "max_gap_s",
    "invalid_arc_node_references",
    "invalid_route_arc_references",
]

issues = inventory[
    (inventory["gps_rows_removed"] > 0)
    | (inventory["non_increasing_timestamp_steps"] > 0)
    | (inventory["invalid_arc_node_references"] > 0)
    | (inventory["invalid_route_arc_references"] > 0)
]

print(f"Trips with one or more flagged integrity issues: {len(issues)}")
display(issues[quality_columns] if len(issues) else pd.DataFrame(columns=quality_columns))


Trips with one or more flagged integrity issues: 0


,partition,trip_id,gps_rows_removed,non_increasing_timestamp_steps,non_1s_intervals,max_gap_s,invalid_arc_node_references,invalid_route_arc_references


In [34]:
display(
    inventory.sort_values("max_gap_s", ascending=False)[
        ["partition", "trip_id", "gps_observations", "non_1s_intervals", "max_gap_s"]
    ].head(15)
)


,partition,trip_id,gps_observations,non_1s_intervals,max_gap_s
72,train,00000090,536,224,153064.0
61,train,00000078,2909,712,52810.0
55,train,00000069,2067,22,2658.0
16,train,00000021,15783,6,2161.0
8,train,00000011,2828,1,909.0
34,train,00000042,7739,81,670.0
64,train,00000081,3224,903,536.0
18,train,00000024,801,23,458.0
36,train,00000044,546,54,341.0
67,train,00000084,16031,50,312.0


## 10. Final output

The `pathsnap_cleaned/` directory now contains:

- `train/<trip_id>/track.csv`
- `train/<trip_id>/nodes.csv`
- `train/<trip_id>/arcs.csv`
- `train/<trip_id>/route.csv`
- The equivalent files under `student_test/`
- `dataset_inventory.csv`
- `partition_summary.csv`

**Important for the next stage:** `.route` is a sequence of reference arc IDs and is not necessarily one label per GPS observation. Candidate-road generation and GPS-to-route alignment must be implemented before creating the supervised classifier training table. Keep the evaluator-testing data separate.
